## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [1]:
import subprocess
result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

tippecanoe v2.80.0



## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [ ]:
from pathlib import Path
import subprocess
import time

input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles")

cmd = [
    "tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

print(result.stderr)   # tippecanoe writes progress to stderr
print(f"\nCompleted in {elapsed:.1f}s")

## Inspecting the Output

In [20]:
from pathlib import Path
input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles") 

size_mb = output_file.stat().st_size / 1_000_000
raw_mb  = input_file.stat().st_size  / 1_000_000

print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")
print(f"Output (PMTiles):      {size_mb:.2f} MB")
print(f"Compression ratio:     {raw_mb / size_mb:.1f}×  smaller")

Input  (raw GeoJSON):  39.6 MB
Output (PMTiles):      360.46 MB
Compression ratio:     0.1×  smaller


## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [18]:
# Use tippecanoe's companion tool to show metadata
result = subprocess.run(
    ["tile-join", "--no-tile-compression", "--if-matched",
     f"--output={output_file.with_suffix('.inspect.pmtiles')}",
     str(output_file)],
    capture_output=True, text=True
)

# Simpler: just show what pmtiles show gives us
result2 = subprocess.run(
    ["pmtiles", "show", str(output_file)],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr or "(install pmtiles CLI: pip install pmtiles)")

pmtiles spec version: 3
tile type: Vector Protobuf (MVT)
bounds: (long: -150.112222, lat: -51.895278) (long: 179.357778, lat: 69.604375)
min zoom: 1
max zoom: 14
center: (long: 13.480225, lat: 52.502847)
center zoom: 14
addressed tiles count: 1280850
tile entries count: 1280762
tile contents count: 1280584
clustered: true
internal compression: 2
tile compression: 2
name ../../data/railroads.pmtiles
description ../../data/railroads.pmtiles
generator_options tippecanoe '--output=../../data/railroads.pmtiles' --force '--minimum-zoom=1' '--maximum-zoom=14' '--simplification=10' --drop-densest-as-needed '--layer=railroads' ../../data/ne_10m_railroads.geojson
antimeridian_adjusted_bounds -150.112222,-51.895278,179.357778,69.604375
tilestats <object...>
format pbf
type overlay
version 2
generator tippecanoe v2.80.0
vector_layers <object...>



## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [ ]:
# Try loading with localtileserver if available
try:
    from localtileserver import TileClient, get_leaflet_tile_layer
    from ipyleaflet import Map

    client = TileClient(str(output_file))
    layer  = get_leaflet_tile_layer(client)
    m = Map(center=client.center(), zoom=client.default_zoom)
    m.add(layer)
    m
except ImportError:
    print("localtileserver not installed.")
    print("Install with: pip install localtileserver")
    print()
    print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
    print(f"File location: {output_file.resolve()}")

## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [ ]:
# Run tippecanoe with --maximum-zoom=8 and compare output size
# Your code here

import subprocess
import os

cmd_exercise_a = [
    "tippecanoe",
    "--output=../../data/railroads_z8.pmtiles",
    "--minimum-zoom=0",
    "--maximum-zoom=8",       # Limiting zoom to 8
    "--simplification=10",    # Keeping our simplification factor
    "--drop-densest-as-needed",
    "--force",
    "../../data/ne_10m_railroads.geojson"
]

print("Baking tiles for Exercise A...")
subprocess.run(cmd_exercise_a)

raw_size = os.path.getsize("../../data/ne_10m_railroads.geojson") / (1024 * 1024)
new_size = os.path.getsize("../../data/railroads_z8.pmtiles") / (1024 * 1024)

print(f"Original GeoJSON: {raw_size:.2f} MB")
print(f"New PMTiles (Z8): {new_size:.2f} MB")
print(f"Reduction: {((1 - (new_size / raw_size)) * 100):.1f}% smaller")
# The output file is now much smaller but we lose the ability to zoom in beyond level 8 and still
# see the proper detailed railroads. We also lose the extrafine detail at that level zoom and ultimately
# we lose to ability to see accurate railroads.

Baking tiles for Exercise A...


For layer 0, using name "ne_10m_railroads"
25413 features, 5899421 bytes of geometry and attributes, 167062 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  8/144/94   


Original GeoJSON: 37.76 MB
New PMTiles (Z8): 6.81 MB
Reduction: 82.0% smaller


## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [ ]:
# Calculate simplification tolerance in meters at different zoom levels
# Compare to our Module 02 epsilon choices
# Your code here

zoom_levels = [2, 5, 8, 12, 14]
base_res_z14 = 0.6 

print(f"{'Zoom':<6} | {'Res (m/px)':<12} | {'Simplification (10px in meters)':<25}")
print("-" * 50)

for z in zoom_levels:
    # Resolution doubles for every zoom level out from 14
    res = base_res_z14 * (2 ** (14 - z))
    error_margin = res * 10
    print(f"{z:<6} | {res:<12.2f} | {error_margin:<25.2f}m")
    # In module 02, we chose epsilon values that were in degrees. Tippecanoe works in meters, and it 
    # automatically calculates the appropriate simplification tolerance based on the zoom level 
    # and the tile pixel size. The simplification tolerance in meters increases as we zoom out, which means
    # that at lower zoom levels, the features will be simplified more aggressively, while at higher zoom levels,
    # they will retain more detail.

Zoom   | Res (m/px)   | Simplification (10px in meters)
--------------------------------------------------
2      | 2457.60      | 24576.00                 m
5      | 307.20       | 3072.00                  m
8      | 38.40        | 384.00                   m
12     | 2.40         | 24.00                    m
14     | 0.60         | 6.00                     m


## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

Tippecanoe decides based on a size limit in each tile, if its too much it begins to simplify that tile. This is different that our scalerank filter because that was something predefined within the data whereas this is decided algorithmically. The manual scalerank approach is less principled it depends on what the creater of the file has decided to be most important to preserve while the tippecanoe method is mathematically based and therefore more principled. The tradeoffs are that in the scalerank method the designer is more in charge of what data is considered important and can make sure specific railroads are preserved, but in the tippecanoe method there will always be a well distributed map that is calculated easier.
---

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?